# TenneT 小时数据：完整分析与 24 小时批量预测

此 notebook 以同目录 `tennet_pipeline.py` 为实现。摘要单元已执行；完整图表与解释请参阅 `analysis_report_zh.html`。所有预测共用每日 UTC 零点的信息截止时间。

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import numpy as np
import joblib
BASE = Path.cwd()
if not (BASE / 'tennet_pipeline.py').exists():
    BASE = BASE / 'outputs'
sys.path.insert(0, str(BASE))
import tennet_pipeline as pipeline
RESULTS = BASE / 'results'
DATA = Path('/Users/corinne_777/Desktop/electric_proj/tennet_hourly_2015-01-01_2022-12-31.csv')
MAPPING = Path('/Users/corinne_777/Desktop/electric_proj/tennet_series_mapping.csv')
print('结果目录:', RESULTS)


结果目录: /Users/corinne_777/Documents/Codex/2026-11-03/squarepoint-dataset-project-2026-10-31/outputs/results


## 1. 完整重跑开关
修改 DATA、MAPPING 到你的输入路径。默认使用已保存结果；设为 True 会重新执行所有分析、模型比较与最终重训。最终测试应在模型选择锁定后再看，避免反复针对测试调参。

In [2]:
RUN_FULL_PIPELINE = False
if RUN_FULL_PIPELINE:
    import subprocess
    subprocess.run([sys.executable, str(BASE/'tennet_pipeline.py'), '--data', str(DATA),
                    '--mapping', str(MAPPING), '--out', str(RESULTS)], check=True)
print('完整 pipeline 开关:', RUN_FULL_PIPELINE)


完整 pipeline 开关: False


## 2. 字段字典与 datetime
Parquet 保留带时区 datetime。CSV 只能保存文本，重新读取时必须 `pd.to_datetime(..., utc=True)`。柏林时间用 `tz_convert` 转换，不删除夏令时重复墙钟小时。

In [3]:
dictionary = json.loads((RESULTS/'column_dictionary.json').read_text())
print(dictionary['id_to_english'])
data = pd.read_parquet(RESULTS/'cleaned_hourly.parquet').drop(columns='datetime_berlin')
print('形状:', data.shape, '索引类型:', data.index.dtype)
print('相邻 UTC 时间差:', data.index.to_series().diff().value_counts().to_dict())
quality = json.loads((RESULTS/'data_quality.json').read_text())
print('缺失数:', quality['missing_values'], '缺失小时:', quality['missing_timestamp_rows'])


{'1223': 'Lignite', '4069': 'Hard coal', '4071': 'Natural gas', '1227': 'Other conventional energy sources', '1224': 'Nuclear', '4067': 'Onshore wind', '1225': 'Offshore wind', '4068': 'Photovoltaics', '1226': 'Hydropower', '4070': 'Pumped storage generation', '4066': 'Biomass', '1228': 'Other renewables', '410': 'Total load', '4359': 'Residual load', '4387': 'Pumped storage consumption'}
形状: (70128, 15) 索引类型: datetime64[us, UTC]
相邻 UTC 时间差: {Timedelta('0 days 01:00:00'): 70127}
缺失数: 0 缺失小时: 0


## 3. 每个变量的季节性
partial R² ≥ 0.02 且跨年验证 SSE 改善，标为 supported。它是探索性证据规则，不是显著性检验。趋势和周期拟合只使用早期训练数据。

In [4]:
season = pd.read_csv(RESULTS/'seasonality_by_variable.csv')
print(season.pivot(index='variable', columns='season', values='evidence').to_string())


season                                     annual           daily          weekly
variable                                                                         
biomass                                 supported       supported            weak
hard_coal                               supported       supported  in_sample_only
hydropower                              supported            weak            weak
lignite                            in_sample_only            weak  in_sample_only
natural_gas                             supported       supported       supported
nuclear                                 supported            weak            weak
offshore_wind                           supported            weak            weak
onshore_wind                            supported            weak            weak
other_conventional_energy_sources  in_sample_only            weak            weak
other_renewables                        supported       supported            weak
photovoltaics   

![周期证据](results/figures/03_seasonality_evidence.png)

## 4. 共线性与特征筛选
VIF 是相对其余 predictors 的诊断。含 response 的原变量矩阵用于检查代数关系；实际预测不能使用同小时未来观测。最终保留特征以 selected_features.csv 为准。

In [5]:
print(pd.read_csv(RESULTS/'vif_diagnostics.csv').round(2).to_string(index=False))
print('\n实际模型矩阵:')
print((RESULTS/'model_feature_collinearity.json').read_text())
print(pd.read_csv(RESULTS/'selected_features.csv').to_string(index=False))


                         variable  VIF_all_15_including_response  VIF_predictors_only  VIF_adjusted_predictors
                          lignite                           1.56                 1.56                     1.17
                        hard_coal                           3.69                 3.69                     2.30
                      natural_gas                           2.45                 2.45                     1.49
other_conventional_energy_sources                           1.10                 1.10                     1.12
                          nuclear                           1.30                 1.30                     1.18
                     onshore_wind                            inf                 4.32                     7.56
                    offshore_wind                            inf                 2.87                     3.12
                    photovoltaics                            inf                 2.03                     2.45
 

## 5. 从历史构造 24 小时输入，并验证无未来泄漏
`lag24`、`lag168` 按目标小时取值；起点前均值按共同起点取值。将起点以后全部数据改写，特征必须保持不变。

In [6]:
origin = pd.Timestamp('2021-06-01T00:00:00Z')
x_before, metadata = pipeline.build_features(data, origins=pd.DatetimeIndex([origin]))
altered = data.copy()
altered.loc[altered.index >= origin] = 999999
x_after, _ = pipeline.build_features(altered, origins=pd.DatetimeIndex([origin]))
pd.testing.assert_frame_equal(x_before, x_after)
assert len(x_before) == 24
print('未来数据改写不改变预测特征：PASS')
print(metadata[['origin_utc','target_utc','horizon']].head().to_string(index=False))


未来数据改写不改变预测特征：PASS
               origin_utc                target_utc  horizon
2021-06-01 00:00:00+00:00 2021-06-01 00:00:00+00:00        1
2021-06-01 00:00:00+00:00 2021-06-01 01:00:00+00:00        2
2021-06-01 00:00:00+00:00 2021-06-01 02:00:00+00:00        3
2021-06-01 00:00:00+00:00 2021-06-01 03:00:00+00:00        4
2021-06-01 00:00:00+00:00 2021-06-01 04:00:00+00:00        5


## 6. 模型选择与测试
2020、2021 H1、2021 H2 三折按时间扩展，按小时数加权 MAE 选模型。2022 用于独立评估。只展示实际已计算结果；完整模型参数在 pipeline.candidates() 和 pipeline.fit_model() 中。

In [7]:
print(pd.read_csv(RESULTS/'cv_ranking.csv').round(3).to_string(index=False))
print('\n最终测试:')
print(pd.read_csv(RESULTS/'test_scores.csv').round(3).to_string(index=False))


                   model  weighted_CV_MAE  weighted_CV_RMSE
      Boost_load_only_31          419.556           588.929
            Boost_all_31          423.622           600.652
            Boost_all_15          443.053           629.197
      Boost_load_only_15          450.649           632.239
       Boost_selected_31          461.150           677.165
Boost_weekly_residual_15          498.224           695.132
               Ridge_100          560.794           767.828
                 Ridge_1          563.622           769.667
             Ridge_10000          597.018           843.038
            ElasticNet_1          621.568           872.882
     Naive_previous_week          705.526          1112.271
           ElasticNet_10         1138.076          1378.678
      Naive_previous_day         1224.548          1862.588

最终测试:
                   model  selected_before_test    n      MAE     RMSE  WAPE_pct    R2  MASE_weekly    bias
      Boost_load_only_31                  True

![预测诊断](results/figures/07_forecast_diagnostics.png)

## 7. 加载模型，单次生成 24 小时预测
生产模型仅用于其训练结束之后的起点。数据末端演示起点是 2022-12-31 UTC，模型训练目标止于前一天；不将它混同于独立 2022 测试。

In [8]:
model = joblib.load(RESULTS/'production_model.joblib')
origin = pd.Timestamp('2022-12-31T00:00:00Z')
history = data.loc[data.index < origin]
future_x, future_meta = pipeline.build_features(history, origins=pd.DatetimeIndex([origin]))
future_meta['forecast'] = pipeline.predict(model, future_x)
saved = pd.read_csv(RESULTS/'endpoint_24h_forecast_demo.csv')
np.testing.assert_allclose(future_meta.forecast, saved.prediction, rtol=1e-12)
print('模型重载预测一致性：PASS')
print(future_meta[['target_utc','target_berlin','horizon','forecast']].to_string(index=False))


模型重载预测一致性：PASS
               target_utc             target_berlin  horizon     forecast
2022-12-31 00:00:00+00:00 2022-12-31 01:00:00+01:00        1 11899.872711
2022-12-31 01:00:00+00:00 2022-12-31 02:00:00+01:00        2 11354.270448
2022-12-31 02:00:00+00:00 2022-12-31 03:00:00+01:00        3 11385.857861
2022-12-31 03:00:00+00:00 2022-12-31 04:00:00+01:00        4 11797.833052
2022-12-31 04:00:00+00:00 2022-12-31 05:00:00+01:00        5 11904.596264
2022-12-31 05:00:00+00:00 2022-12-31 06:00:00+01:00        6 11929.845995
2022-12-31 06:00:00+00:00 2022-12-31 07:00:00+01:00        7 12561.667880
2022-12-31 07:00:00+00:00 2022-12-31 08:00:00+01:00        8 13638.132538
2022-12-31 08:00:00+00:00 2022-12-31 09:00:00+01:00        9 14391.129607
2022-12-31 09:00:00+00:00 2022-12-31 10:00:00+01:00       10 14573.868821
2022-12-31 10:00:00+00:00 2022-12-31 11:00:00+01:00       11 14805.830700
2022-12-31 11:00:00+00:00 2022-12-31 12:00:00+01:00       12 14765.315126
2022-12-31 12:00:00+00:

## 8. 缺失值实验及限制
原始数据无缺失。人工遮盖实验只对负荷历史进行，必须同时看 coverage 和 MAE；不代表每个能源的最佳插补方法。未来标签缺失不插补用于训练/评分。经验 90% 预测带也不保证未来达到 90% 覆盖。

In [9]:
print(pd.read_csv(RESULTS/'simulated_missingness_load.csv').round(3).to_string(index=False))
summary = json.loads((RESULTS/'model_summary.json').read_text())
print('2022经验区间覆盖率:', summary['interval_coverage90'])
print('残差自相关:', summary['residual_acf'])


 gap_hours                        method  masked_hours  coverage      MAE
         1                  forward_fill            30     1.000  708.708
         1                  previous_day            30     1.000 1493.117
         1                 previous_week            30     1.000  872.983
         1 bounded_forward_then_seasonal            30     1.000  708.708
         3                  forward_fill            90     1.000 1377.519
         3                  previous_day            90     1.000 1209.858
         3                 previous_week            90     0.967  889.310
         3 bounded_forward_then_seasonal            90     1.000 1052.244
        24                  forward_fill           702     1.000 2865.705
        24                  previous_day           702     0.957 1428.686
        24                 previous_week           702     0.981 1006.772
        24 bounded_forward_then_seasonal           702     1.000  968.828
       168                  forward_fi